# NB-09: CausalConv3d, ResidualBlock, and AttentionBlock

## Learning Objectives
- Understand asymmetric temporal padding in `CausalConv3d` — why left-only zero-padding prevents future frame leakage and how the 6-tuple `_padding = (right_W, right_W, bot_H, bot_H, 2*t_pad, 0)` is derived from the kernel size
- Trace `ResidualBlock`'s full sequential pipeline layer by layer — `RMSNorm → SiLU → CausalConv3d → RMSNorm → SiLU → Dropout → CausalConv3d` — and see both skip connection paths (Identity when `in_dim==out_dim`, learned 1×1 `CausalConv3d` when dimensions change)
- See how `AttentionBlock` collapses the temporal dimension into the batch dimension to apply per-frame spatial self-attention, and verify that the zero-initialized `proj.weight` means attention output is discarded at initialization (gradients flow through the skip connection)

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm concept — the VAE has its own `RMS_norm` variant with different broadcasting; this notebook shows the difference)
- **Assumed concepts:** `nn.Conv3d`, `F.pad`, residual connections, scaled dot-product attention

## Concept Map

```
CausalConv3d
  └─> used in: ResidualBlock (both conv layers + optional shortcut)
  └─> used in: every Resample in Encoder3d/Decoder3d (NB-10, NB-11)
  └─> used in: conv1 head of Encoder3d (NB-10), conv2 input of Decoder3d (NB-11)

ResidualBlock
  └─> stacked in every encoder/decoder level (NB-10, NB-11)
  └─> used in middle block (between encoder levels and decoder levels)

AttentionBlock
  └─> used in encoder/decoder middle blocks (NB-10, NB-11)
  └─> pattern: Middle = ResidualBlock + AttentionBlock + ResidualBlock
```


In [ ]:
import sys
import types
import importlib.util
import pathlib

# Stub tqdm BEFORE loading wan_video_vae.py (which imports tqdm at module level)
_tqdm_stub = types.ModuleType('tqdm')
_tqdm_stub.tqdm = lambda x, **kw: x
sys.modules['tqdm'] = _tqdm_stub

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):  # search up to 10 levels
    if (_candidate / "diffsynth" / "models" / "wan_video_vae.py").exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:  # reached filesystem root
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_vae.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Load wan_video_vae.py directly, bypassing the broken diffsynth/__init__.py chain
_vae_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_vae.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_vae", _vae_path)
vae = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_vae"] = vae
_spec.loader.exec_module(vae)

from diffsynth.models.wan_video_vae import (
    CausalConv3d, RMS_norm, ResidualBlock, AttentionBlock,
    Resample, Encoder3d, Decoder3d, VideoVAE_, patchify, unpatchify
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

print("Setup complete.")

## 1. CausalConv3d — Causal Temporal Padding for Video

Video generation models need convolutions that don't look at future frames. A standard `nn.Conv3d` with symmetric padding of `(1,1,1)` would see 1 frame in the future for every output frame, creating information leakage from frames that haven't been generated yet.

`CausalConv3d` solves this with **asymmetric temporal padding**: prepend `2 * temporal_pad` zero frames on the LEFT side of the temporal dimension and zero frames on the RIGHT. Each output frame can only depend on current and past input frames — no future leakage.

The class inherits from `nn.Conv3d`, overrides the padding mechanism in `__init__`, and applies `F.pad` manually before calling `super().forward(x)`. The key is the 6-tuple `_padding` which `F.pad` applies in the order `(left_W, right_W, top_H, bot_H, front_T, back_T)`.

Source: `diffsynth/models/wan_video_vae.py`, lines 33–52

### Asymmetric Temporal Padding

```
CausalConv3d Temporal Padding  (kernel_t=3, padding=1 -> temporal_pad=1)
=========================================================================
Input frames:   [f-2] [f-1] [ f ]   <- T=3 real frames
                  |     |     |
F.pad prepend:  [  0] [  0] [f-2] [f-1] [f] <- 2*temporal_pad = 2 zeros prepended
                  |     |     |     |     |
Conv3d k=3:          [window-0]          = uses f-2, f-1, f  (no future!)
                            [window-1]   = uses f-1,  f, 0  <- right pad = 0
No right padding: output T = input T (shape preserved)
Formula: _padding = (right_W, right_W, bot_H, bot_H, 2*t_pad, 0)
         for padding=1: (1, 1, 1, 1, 2, 0)
```


In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 33-52
conv = CausalConv3d(3, 8, kernel_size=3, padding=1)
print(f"_padding tuple: {conv._padding}")
# F.pad order: (left_W, right_W, top_H, bot_H, front_T, back_T)
# So: W gets +-1, H gets +-1, T gets +2 on left and 0 on right

x = torch.randn(1, 3, 4, 8, 8)   # [B, C, T, H, W]
with torch.no_grad():
    out = conv(x)
print(f"Input:  {list(x.shape)}")    # [1, 3, 4, 8, 8]
print(f"Output: {list(out.shape)}")  # [1, 8, 4, 8, 8]
assert out.shape == (1, 8, 4, 8, 8), f"Expected (1,8,4,8,8), got {out.shape}"
# T=4, H=8, W=8 all PRESERVED -- only channels change (3 -> 8)

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 36-39
# After super().__init__(3, 8, kernel_size=3, padding=1):
#   self.padding = (1, 1, 1)  -- (temporal, height, width)
# Then __init__ derives:
#   self._padding = (padding[2], padding[2], padding[1], padding[1], 2*padding[0], 0)
#                 = (1,           1,           1,          1,          2*1,         0)
#                 = (1, 1, 1, 1, 2, 0)
# And sets: self.padding = (0, 0, 0)  -- disables Conv3d's own padding

# Verify the formula for different kernel sizes:
conv_k5 = CausalConv3d(3, 8, kernel_size=5, padding=2)
print(f"k=5, p=2 -> _padding: {conv_k5._padding}")
assert conv_k5._padding == (2, 2, 2, 2, 4, 0)
# temporal_pad = 2 -> prepend 2*2=4 zero frames, 0 on right

conv_k1 = CausalConv3d(3, 8, kernel_size=1, padding=0)
print(f"k=1, p=0 -> _padding: {conv_k1._padding}")
assert conv_k1._padding == (0, 0, 0, 0, 0, 0)
# 1x1x1 conv needs no padding at all

print("Padding formula verified for k=3, k=5, k=1")

## 2. RMS_norm — VAE vs DiT Normalization Convention

Both the VAE and DiT use RMSNorm, but with different broadcasting conventions that reflect their tensor formats. VAE tensors are 5D `[B, C, T, H, W]` (channel-first), so normalization runs over the channel axis `dim=1`. DiT tensors are 3D `[B, S, dim]` (channel-last), so normalization runs over the last axis `dim=-1`. Using the wrong variant on the wrong tensor format produces incorrect normalization.

| Property | VAE `RMS_norm` | DiT `WanRMSNorm` (NB-01) |
|----------|---------------|---------------------------|
| Source | `wan_video_vae.py:55` | `wan_video_dit.py:101` |
| Normalize axis | `dim=1` (channel) | `dim=-1` (feature) |
| Input format | `[B, C, T, H, W]` | `[B, S, dim]` |
| Gamma shape | `[C, 1, 1, 1]` (`images=False`) | `[dim]` |
| API | `RMS_norm(dim, images=False)` | `WanRMSNorm(dim, eps)` |

Source: `diffsynth/models/wan_video_vae.py`, lines 55–70

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 55-70
norm = RMS_norm(dim=8, channel_first=True, images=False)
print(f"gamma shape: {norm.gamma.shape}")  # [8, 1, 1, 1] -- broadcasts over T, H, W

x = torch.randn(1, 8, 4, 8, 8)  # [B, C, T, H, W]
out = norm(x)
assert out.shape == x.shape  # [1, 8, 4, 8, 8]
print(f"RMS_norm: {list(x.shape)} -> {list(out.shape)}")
# images=False -> gamma [C, 1, 1, 1] for 5D tensors
# images=True  -> gamma [C, 1, 1] for 4D tensors (used by AttentionBlock after T collapse)

norm_images = RMS_norm(dim=8, channel_first=True, images=True)
print(f"gamma shape (images=True):  {norm_images.gamma.shape}")  # [8, 1, 1] -- for 4D tensors

## 3. ResidualBlock — The VAE's Workhorse Module

`ResidualBlock` is the fundamental building block stacked at every level of the encoder and decoder. It applies two `CausalConv3d` layers with normalization and activation in between, then adds a skip connection.

**Sequential pipeline** (stored in `self.residual`):
1. `RMS_norm(in_dim, images=False)` — normalize the input channels
2. `SiLU()` — smooth activation (sigmoid linear unit)
3. `CausalConv3d(in_dim, out_dim, 3, padding=1)` — first spatial-temporal conv
4. `RMS_norm(out_dim, images=False)` — normalize again at out_dim channels
5. `SiLU()` — second activation
6. `Dropout(dropout)` — regularization (default 0.0, no-op in eval)
7. `CausalConv3d(out_dim, out_dim, 3, padding=1)` — second spatial-temporal conv

**Skip connection** (`self.shortcut`):
- `in_dim == out_dim`: `nn.Identity()` — pass the input through unchanged
- `in_dim != out_dim`: `CausalConv3d(in_dim, out_dim, 1)` — learned 1×1×1 channel projection

**Forward:** `return x_residual + shortcut(x_input)` — the skip connection adds the (optionally projected) input to the residual path output.

Source: `diffsynth/models/wan_video_vae.py`, lines 267–301

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 267-301
rb = ResidualBlock(in_dim=8, out_dim=8)
print("ResidualBlock(in_dim=8, out_dim=8)")
print(f"  shortcut: {type(rb.shortcut).__name__}")  # Identity (in_dim == out_dim)
print("  residual pipeline:")
for i, layer in enumerate(rb.residual):
    print(f"    [{i}] {layer}")
# Expected output:
#   [0] RMS_norm   -- [B,8,T,H,W] -> [B,8,T,H,W]
#   [1] SiLU()     -- activation
#   [2] CausalConv3d(8, 8, ...)  -- [B,8,T,H,W] -> [B,8,T,H,W]
#   [3] RMS_norm
#   [4] SiLU()
#   [5] Dropout(p=0.0)           -- no dropout by default
#   [6] CausalConv3d(8, 8, ...)  -- [B,8,T,H,W] -> [B,8,T,H,W]

In [ ]:
# Path 1: same dimensions -- shortcut is Identity
rb_same = ResidualBlock(in_dim=8, out_dim=8)
print(f"Same dim shortcut: {type(rb_same.shortcut).__name__}")  # Identity

# Path 2: different dimensions -- shortcut is 1x1 CausalConv3d
rb_proj = ResidualBlock(in_dim=8, out_dim=16)
print(f"Diff dim shortcut: {type(rb_proj.shortcut).__name__}")  # CausalConv3d
print(f"  1x1 conv: {rb_proj.shortcut}")

x = torch.randn(1, 8, 4, 8, 8)  # [B, C_in=8, T, H, W]
with torch.no_grad():
    out_same = rb_same(x)   # [1, 8, 4, 8, 8]  -- channels unchanged
    out_proj = rb_proj(x)   # [1, 16, 4, 8, 8] -- channels projected 8->16

assert out_same.shape == (1, 8, 4, 8, 8), f"Expected (1,8,4,8,8), got {out_same.shape}"
assert out_proj.shape == (1, 16, 4, 8, 8), f"Expected (1,16,4,8,8), got {out_proj.shape}"
print(f"Same dim: {list(x.shape)} -> {list(out_same.shape)}")
print(f"Diff dim: {list(x.shape)} -> {list(out_proj.shape)}")
# forward: return x_residual + h  where h = shortcut(x_input)
# When in_dim != out_dim: shortcut projects input to match residual output channels

## 4. AttentionBlock — Per-Frame Spatial Self-Attention

`AttentionBlock` applies single-head self-attention **within each frame** (spatial positions attend to each other). The key design decision: the temporal dimension `T` is folded into the batch dimension before attention, so frames do **not** attend to each other. This is 2D spatial attention on 5D video data.

**Forward pass step by step:**
1. Save identity: `identity = x`  — for the skip connection
2. Rearrange: `'b c t h w -> (b t) c h w'` — collapses T=4 frames into batch: `[B,C,T,H,W] -> [B*T,C,H,W]`
3. Normalize: `self.norm(x)` — `RMS_norm(dim, images=True)`, gamma `[C,1,1]` for 4D tensors
4. Project to Q, K, V: `self.to_qkv(x)` — `Conv2d(dim, dim*3, 1)` maps each spatial position to Q+K+V
5. Reshape and chunk: `q, k, v` each shape `[B*T, 1, H*W, dim]` — single head, H*W spatial tokens
6. Attention: `F.scaled_dot_product_attention(q, k, v)` — each frame attends to its own H*W positions
7. Output projection: `self.proj(x)` — `Conv2d(dim, dim, 1)`, weight **zero-initialized**, bias is non-zero
8. Rearrange back: `'(b t) c h w -> b c t h w'` — restore temporal dimension
9. Return: `x + identity` — residual connection

**Zero-initialized `proj.weight`:** Source code (`line 319`) zeroes only `proj.weight`, not `proj.bias`. This means at initialization `proj(features) = 0 * features + bias = bias` — the attention output is discarded (replaced by a constant). Gradients flow through the skip connection from the start of training, allowing the model to gradually learn to use the attention path.

Source: `diffsynth/models/wan_video_vae.py`, lines 304–342

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 304-342
ab = AttentionBlock(dim=8)
print("AttentionBlock(dim=8)")
print(f"  norm: {ab.norm}")           # RMS_norm(8, images=True) -- gamma [8,1,1]
print(f"  to_qkv: {ab.to_qkv}")      # Conv2d(8, 24, 1) -- projects to q,k,v
print(f"  proj: {ab.proj}")           # Conv2d(8, 8, 1) -- output projection

# Source: diffsynth/models/wan_video_vae.py, line 319: nn.init.zeros_(self.proj.weight)
# Only weight is zeroed; bias uses nn.Conv2d default initialization (non-zero)
print(f"  proj.weight all zeros: {(ab.proj.weight == 0).all().item()}")   # True
print(f"  proj.bias all zeros:   {(ab.proj.bias == 0).all().item()}")     # False -- bias not zeroed

x = torch.randn(1, 8, 4, 8, 8)  # [B=1, C=8, T=4, H=8, W=8]
with torch.no_grad():
    out = ab(x)

# Step-by-step shape trace:
# 1. rearrange 'b c t h w -> (b t) c h w': [1,8,4,8,8] -> [4,8,8,8]  (T folded into batch)
# 2. norm: [4,8,8,8] -> [4,8,8,8]
# 3. to_qkv: [4,8,8,8] -> [4,24,8,8]  (dim*3 = 24)
# 4. reshape+permute+chunk: q,k,v each [4,1,64,8]  (h*w=64, single head, dim=8)
# 5. scaled_dot_product_attention: [4,1,64,8]
# 6. reshape back: [4,8,8,8]
# 7. proj: [4,8,8,8] -> [4,8,8,8]  (weight=0: proj(x) = bias, discards attention output)
# 8. rearrange '(b t) c h w -> b c t h w': [4,8,8,8] -> [1,8,4,8,8]
# 9. return x + identity

assert out.shape == (1, 8, 4, 8, 8), f"Expected (1,8,4,8,8), got {out.shape}"
print(f"AttentionBlock: {list(x.shape)} -> {list(out.shape)}")
print("Each frame (T=4) attends to its own H*W=64 spatial positions independently")

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, line 319 -- nn.init.zeros_(self.proj.weight)
# Verify: weight IS zero-initialized; attention output discarded at init
ab_fresh = AttentionBlock(dim=8)

# proj.weight is zeroed => attention features are discarded (proj output = bias only)
assert (ab_fresh.proj.weight == 0).all(), "proj.weight must be zero at init"
print(f"proj.weight all zeros: {(ab_fresh.proj.weight == 0).all().item()}")  # True

# proj.bias is NOT zeroed -- it uses nn.Conv2d default (Kaiming uniform scaled)
# So proj(features) = 0 * features + bias = constant-per-channel bias
print(f"proj.bias all zeros:   {(ab_fresh.proj.bias == 0).all().item()}")   # False

# At initialization: output = skip(input) + bias_constant_per_channel
# This is close to identity but not exact (bias adds a small spatial-uniform offset)
# Key benefit: weight=0 prevents attention features from disturbing skip gradient flow
x_test = torch.randn(1, 8, 2, 4, 4)  # [B, C, T, H, W]
with torch.no_grad():
    out_test = ab_fresh(x_test)
assert out_test.shape == x_test.shape, f"Shape must be preserved: {out_test.shape}"
print(f"AttentionBlock output shape preserved: {list(out_test.shape)}")
print("Zero-weight proj -> attention output discarded at init; gradients flow via skip connection")

## Summary

### Key Takeaways

- **CausalConv3d**: wraps `nn.Conv3d` with asymmetric temporal padding `_padding = (right_W, right_W, bot_H, bot_H, 2*t_pad, 0)`. Preserves T, H, W dimensions. Ensures each output frame sees only current and past input frames — no future leakage.

- **RMS_norm**: VAE uses channel_first convention (`dim=1`, gamma `[C,1,1,1]` for 5D or `[C,1,1]` for 4D tensors), different from DiT's channel_last (`dim=-1`, gamma `[dim]`). Both are RMSNorm but with different broadcasting conventions matched to their tensor formats.

- **ResidualBlock**: pipeline of `RMSNorm → SiLU → CausalConv3d → RMSNorm → SiLU → Dropout → CausalConv3d` with Identity or 1×1 skip connection. The building block stacked in every encoder/decoder level (NB-10, NB-11).

- **AttentionBlock**: per-frame spatial self-attention (T collapsed into batch). Single-head. `proj.weight` is zero-initialized so attention output is discarded at init; `proj.bias` is non-zero (standard Conv2d init). The skip connection carries gradients from the start of training.

### Source References

| Symbol | Location |
|--------|----------|
| `CausalConv3d` | `diffsynth/models/wan_video_vae.py`, line 33 |
| `RMS_norm` | `diffsynth/models/wan_video_vae.py`, line 55 |
| `ResidualBlock` | `diffsynth/models/wan_video_vae.py`, line 267 |
| `AttentionBlock` | `diffsynth/models/wan_video_vae.py`, line 304 |


## Exercises

### Exercise 1 — Change CausalConv3d kernel size
Create `CausalConv3d(3, 8, kernel_size=5, padding=2)` and print `_padding`. How does the temporal prepend change compared to `kernel_size=3, padding=1`? Run a forward pass with `x = torch.randn(1, 3, 4, 8, 8)` and verify that the output shape is still `[1, 8, 4, 8, 8]` — T, H, W are preserved regardless of kernel size.

### Exercise 2 — ResidualBlock channel expansion
Create `ResidualBlock(in_dim=8, out_dim=32)` and inspect the shortcut. What is the 1×1 `CausalConv3d`'s weight shape? (Hint: `CausalConv3d(in, out, 1)` has weight shape `[out, in, 1, 1, 1]`.) Run a forward pass with `x = torch.randn(1, 8, 4, 8, 8)` and verify the output has 32 channels.

### Exercise 3 — AttentionBlock parameter count
Count total parameters in `AttentionBlock(dim=8)`: `sum(p.numel() for p in ab.parameters())`. How many are in `to_qkv` vs `proj`? `to_qkv = Conv2d(8, 24, 1)` has `8*24*1*1 + 24 = 216` parameters. `proj = Conv2d(8, 8, 1)` has `8*8*1*1 + 8 = 72` parameters. What fraction of the block is the zero-initialized weight output projection?
